In [45]:
import pandas as pd
import re

df=pd.read_csv("train.csv")

mots_inutiles = set([
    "i","me","my","myself","we","our","ours","ourselves",
    "you","your","yours","yourself","yourselves",
    "he","him","his","himself",
    "she","her","hers","herself",
    "it","its","itself",
    "they","them","their","theirs","themselves",
    "what","which","who","whom","this","that","these","those",
    "am","is","are","was","were","be","been","being",
    "have","has","had","do","does","did",
    "a","an","the","and","or","but","if","because","as",
    "until","while","of","at","by","for","with","about","against",
    "between","into","through","during","before","after","above","below",
    "to","from","up","down","in","out","on","off","over","under",
    "again","further","then","once","get", "just", "like", "can", "not","know",
    "want","time","how","when","now","all","feel", "really", "things", "just", "like",
    "very", "too", "lot", "there", "back",
    "even", "every", "years", "told",
    "going", "still", "one", "now"
])
contractions = {
    "don't":"do not",
    "can't":"can not",
    "i'm":"i am",
    "it's":"it is",
    "i've":"i have",
    "you're":"you are",
    "we're":"we are",
    "they're":"they are"
}
def preprocess(texte):
    if pd.isna(texte):
        return []

    texte=texte.lower()
    for k,v in contractions.items():
        texte=texte.replace(k, v)
    texte=re.sub(r'[^\w\s]',' ',texte)

    tokens=texte.split()

    tokens=[t for t in tokens if t not in mots_inutiles]
    tokens=[t for t in tokens if len(t) > 2]

    return tokens

df['patient_clean'] = df.iloc[:,0].apply(preprocess)
df['therapist_clean'] = df.iloc[:,1].apply(preprocess)
nb_patients=df.iloc[:,0].nunique()
print("Patients uniques:", nb_patients)
nb_therapeutes=df.iloc[:,1].nunique()
print("Thérapeutes uniques:", nb_therapeutes)


echange_moyens = len(df) / nb_patients
print("Échanges moyens par patient:", echange_moyens)



Patients uniques: 995
Thérapeutes uniques: 2479
Échanges moyens par patient: 3.52964824120603


In [46]:
from collections import Counter

tout_mots=[mot for tokens in df['patient_clean'] for mot in tokens]
freq = Counter(tout_mots)

print(freq.most_common(20))

[('never', 623), ('always', 509), ('relationship', 504), ('should', 461), ('think', 442), ('people', 434), ('love', 428), ('anxiety', 412), ('other', 406), ('boyfriend', 396), ('help', 392), ('counseling', 390), ('life', 377), ('having', 353), ('issues', 344), ('more', 331), ('need', 331), ('sex', 330), ('family', 326), ('depression', 320)]


In [4]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 48.8 MB/s eta 0:00:00


In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

df['patient_text'] = df['patient_clean'].apply(lambda x: " ".join(x))

df = df.drop_duplicates(subset="patient_text")
df = df.drop_duplicates(subset="patient_clean")

tfidf = TfidfVectorizer(max_features=2000)

X = tfidf.fit_transform(df['patient_text'])
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster_tfidf'] = kmeans.fit_predict(X)
print(df[['patient_text', 'cluster_tfidf']].head(10))
print(df['cluster_tfidf'].value_counts())
print(df[df['cluster_tfidf'] == 0]['patient_text'].head(10))
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = tfidf.get_feature_names_out()

for i in range(5):
    print(f"Cluster {i}")
    top_words = [terms[ind] for ind in order_centroids[i, :10]]
    print(top_words)

                                         patient_text  cluster_tfidf
0   some feelings barely sleep nothing think worth...              2
23  many issues address history sexual abuse breas...              3
70  feeling more more month started having trouble...              1
72  facing severe depression anxiety distracts can...              3
81                        place where content day day              4
84  severe problem major several minor operations ...              3
86  suffer adult adhd anxiety disorder depression ...              0
89  few ago making love wife known reason lost ere...              0
95  struggle depression well pretty intense mood s...              1
99  self harm stop awhile see something sad depres...              3
cluster_tfidf
3    226
1    199
2    162
4    134
0    110
Name: count, dtype: int64
86     suffer adult adhd anxiety disorder depression ...
89     few ago making love wife known reason lost ere...
135    girlfriend quit drinking became dep

In [60]:
from gensim.models import Word2Vec
custom_stopwords = {"feel", "really", "things", "just", "like"}
df['patient_clean'] = df['patient_clean'].apply(
    lambda x: [w for w in x if w not in custom_stopwords]
)
model = Word2Vec(sentences=df['patient_clean'], vector_size=100, window=5, min_count=3,sg=1)
model.wv.most_similar("depression")


[('personality', 0.9982902407646179),
 ('self', 0.9982423186302185),
 ('stress', 0.9981176257133484),
 ('anxiety', 0.9980449676513672),
 ('severe', 0.9979996085166931),
 ('attacks', 0.9978879690170288),
 ('problems', 0.9978703260421753),
 ('diagnosed', 0.9978135228157043),
 ('major', 0.9976251125335693),
 ('dealing', 0.9976032376289368)]

In [57]:
model.wv.most_similar("anxiety")


[('depression', 0.9974012970924377),
 ('stress', 0.9972684383392334),
 ('might', 0.9965754151344299),
 ('suicidal', 0.9965676069259644),
 ('attacks', 0.9964630007743835),
 ('esteem', 0.9964033961296082),
 ('disorder', 0.9963816404342651),
 ('sleeping', 0.9963577389717102),
 ('causing', 0.9963231086730957),
 ('problems', 0.9962871074676514)]

In [58]:
model.wv.most_similar("relationship")

[('experience', 0.9981099367141724),
 ('couple', 0.9980774521827698),
 ('women', 0.9980396628379822),
 ('emotionally', 0.9980055689811707),
 ('older', 0.9979939460754395),
 ('divorce', 0.9979887008666992),
 ('four', 0.9979832172393799),
 ('long', 0.9979612231254578),
 ('constantly', 0.9979497790336609),
 ('nowhere', 0.9979326128959656)]

In [49]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
vectorizer = CountVectorizer(max_features=2000)
X = vectorizer.fit_transform(df['patient_text'])


In [50]:
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)
mots=vectorizer.get_feature_names_out()

for i, sujet in enumerate(lda.components_):
    print(f"Sujet {i}")
    print([mots[i] for i in sujet.argsort()[-10:]])

Sujet 0
['ago', 'family', 'should', 'sex', 'said', 'relationship', 'together', 'boyfriend', 'other', 'love']
Sujet 1
['mother', 'will', 'other', 'disorder', 'always', 'never', 'something', 'child', 'relationship', 'boyfriend']
Sujet 2
['says', 'need', 'should', 'past', 'make', 'think', 'away', 'life', 'people', 'help']
Sujet 3
['parents', 'having', 'wrong', 'talk', 'school', 'see', 'say', 'dad', 'mom', 'always']
Sujet 4
['someone', 'will', 'more', 'friends', 'would', 'family', 'always', 'life', 'never', 'think']
